# Which teacher gives a reliable distribution over visual tokens?

Five candidates, one referee: the cached leave-one-out ground truth
(`wearvqa_faithfulness.pt`), sinks excluded from both sides.

| candidate | what it is | status |
|---|---|---|
| **attention** | current pipeline, `imp_a` over answer rows | 32.8% @k=10 |
| **rollout** (spec Method A) | `A~ = 0.5*A + 0.5*I` chained across layers | **never actually computed** — we only averaged layers |
| **grad x input** (spec Method C) | `abs( d logP(a)/d emb[i] . emb[i] )` | untested |
| **attn x grad** (Method C variant) | `abs( A * dL/dA )` on the answer->image block | untested |
| **diversity** (DivPrune-style) | farthest-point sampling over token embeddings | untested; literature says it beats attention |

**Why these.** Method C is the only untested candidate that approximates the *right target* — it is
a first-order estimate of the ablation, not a routing statistic:

```
log P(a | G) - log P(a | G with token i removed)  ~=  grad_{emb[i]} log P(a) . emb[i]
```

Rollout is what the FRM spec recommends for scale and we have never run it. Diversity is what the
published comparisons say actually wins, and is the one arm that could invalidate the whole
importance-based framing.

**Perturbation mismatch, deliberately handled.** Our LOO ablates with `attention_mask -> 0` (the
token becomes invisible), whereas grad x input approximates *zeroing the embedding*. Different
perturbations. So both gradient variants are computed — whichever tracks the cached drops better
also tells us which perturbation the label should be defined against.

Cost: ~20 forward + backward passes. Gaze is not involved, so this is independent of the
aspect-ratio question.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

CACHE = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2.pt"
KS    = (5, 10, 12)
LOSS_SCALE = 1e3        # fp16 backward would underflow without this

data = torch.load(CACHE, weights_only=False)
L_v  = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)
sinks = (torch.load(SINKF, weights_only=False)["sink_mask"].bool()
         if os.path.exists(SINKF)
         else VS.sink_token_mask(VS.make_baseline([d["imp_q"] for d in data]), 7))
cand     = VS.candidate_mask(L_v, exclude=[sinks])
cand_idx = torch.nonzero(cand, as_tuple=False).squeeze(-1)
n_cand   = int(cand.sum())

model, processor, device = S._load_smolvlm("HuggingFaceTB/SmolVLM2-2.2B-Instruct")
tokenizer = processor.tokenizer
print(f"{N} examples | L_v={L_v} | candidates {n_cand} | "
      + ", ".join(f"chance@{k}={k/n_cand:.1%}" for k in KS))

## 2. Machinery

Gradients are taken on the **input to decoder layer 0** (the merged image+text embeddings) and on
the **attention probabilities**, both captured with hooks so nothing in `transformers` needs editing.

In [ ]:
def find_decoder_layers(model, n_expected):
    """The LLM's decoder-layer ModuleList (the vision tower has a different depth)."""
    hits = [(name, m) for name, m in model.named_modules()
            if isinstance(m, torch.nn.ModuleList) and len(m) == n_expected]
    if not hits:
        raise RuntimeError(f"no ModuleList of length {n_expected}")
    for name, m in hits:                      # prefer the language side if ambiguous
        if any(t in name for t in ("text", "language", "llm")):
            return m
    return hits[0][1]


def grad_capturing_eager():
    """Like the probe's eager attention, but keeps probs IN the graph and retains its grad."""
    try:
        from transformers.models.llama.modeling_llama import repeat_kv
    except Exception:
        def repeat_kv(h, n):
            b, kv, s, d = h.shape
            return h if n == 1 else h[:, :, None].expand(b, kv, n, s, d).reshape(b, kv * n, s, d)

    def fn(module, query, key, value, attention_mask=None, scaling=None, dropout=0.0, **kw):
        if scaling is None:
            scaling = getattr(module, "scaling", 1.0)
        ng = getattr(module, "num_key_value_groups", 1)
        k, v = repeat_kv(key, ng), repeat_kv(value, ng)
        attn = torch.matmul(query, k.transpose(2, 3)) * scaling
        if attention_mask is not None:
            attn = attn + attention_mask[:, :, :, : k.shape[-2]]
        probs = F.softmax(attn, dim=-1, dtype=torch.float32).to(query.dtype)
        if probs.requires_grad:
            probs.retain_grad()
        module._probs = probs                       # kept IN the graph (not detached)
        out = torch.matmul(F.dropout(probs, p=dropout, training=module.training), v)
        return out.transpose(1, 2).contiguous(), None
    return fn


def build_inputs(image, question, answer):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])


def rollout(post, ans_rows, img_cols):
    """Method A: R = A~(L) @ ... @ A~(1), A~ = 0.5*A_headmean + 0.5*I, rows renormalised."""
    layers = sorted(post)
    L = post[layers[0]].shape[-1]
    eye = torch.eye(L)
    R = None
    for l in layers:
        A = post[l].mean(0)                       # [L, L] head-average
        At = 0.5 * A + 0.5 * eye
        At = At / At.sum(-1, keepdim=True).clamp_min(1e-12)
        R = At if R is None else At @ R
    return R[ans_rows][:, img_cols].sum(0)        # [L_v]


def farthest_point(emb, n=None):
    """DivPrune-style: score = reverse selection order under cosine farthest-point sampling."""
    n = n or emb.shape[0]
    x = F.normalize(emb.float(), dim=-1)
    sel = [int((x @ x.mean(0)).argmax())]          # seed: most central token
    mind = 1 - x @ x[sel[0]]
    for _ in range(n - 1):
        mind[sel[-1]] = -1
        nxt = int(mind.argmax()); sel.append(nxt)
        mind = torch.minimum(mind, 1 - x @ x[nxt])
    score = torch.zeros(emb.shape[0])
    for rank, i in enumerate(sel):
        score[i] = len(sel) - rank                 # earlier pick = higher score
    return score

print("ok")

## 3. Run — one forward + backward per example

In [ ]:
n_layers_expected = 24          # from the layer-band sweep; adjust if the model changes
dec_layers = find_decoder_layers(model, n_layers_expected)
print(f"decoder layers found: {len(dec_layers)}")

results, failures = [], defaultdict(int)
for i, d in enumerate(data):
    try:
        img = S.load_image(d["img_path"])
        inp, n_prompt = build_inputs(img, d["question"], d["answer"])
        ids = inp["input_ids"][0].cpu()
        iid = S._find_image_token_id(model, processor)
        pad = tokenizer.pad_token_id
        img_mask = ids == iid
        txt_mask = (ids != iid) & (ids != (pad if pad is not None else -10**9))
        img_cols = torch.nonzero(img_mask).squeeze(-1)
        tpos     = torch.nonzero(txt_mask).squeeze(-1)
        ans_rows = torch.tensor([int(p) for p in tpos.tolist() if int(p) >= n_prompt])

        store = {}
        def pre_hook(_m, args):
            h = args[0]
            if h.requires_grad:
                h.retain_grad(); store["h"] = h
            return None
        hh = dec_layers[0].register_forward_pre_hook(pre_hook)
        patched = S._patch_eager_globals(grad_capturing_eager())
        try:
            model.zero_grad(set_to_none=True)
            out = model(**inp)
            logits = out.logits[0].float()
            lp  = torch.log_softmax(logits[:-1], dim=-1)
            tgt = inp["input_ids"][0, 1:]
            logp = lp.gather(-1, tgt[:, None]).squeeze(-1)[n_prompt - 1:].sum()
            (logp * LOSS_SCALE).backward()
        finally:
            S._unpatch_eager_globals(patched); hh.remove()

        rec = {}

        # --- grad x input on the merged embeddings (Method C, spec form) ---
        if "h" in store and store["h"].grad is not None:
            h = store["h"][0].detach().float(); gr = store["h"].grad[0].detach().float()
            rec["grad x input"] = (gr[img_cols] * h[img_cols]).sum(-1).abs().cpu()
            rec["_emb"] = h[img_cols].cpu()
        else:
            failures["grad x input"] += 1

        # --- attn x grad, and the post-softmax maps for rollout / attention ---
        post, ag = {}, torch.zeros(len(img_cols))
        got_ag = False
        for m in model.modules():
            p = getattr(m, "_probs", None)
            if p is None:
                continue
            if p.shape[-1] == len(ids) and p.shape[-2] == len(ids):
                li = int(getattr(m, "layer_idx", len(post)))
                post[li] = p[0].detach().float().cpu()
                if p.grad is not None:
                    blk = (p[0].detach() * p.grad[0].detach()).abs().float()
                    ag += blk[:, ans_rows][:, :, img_cols].sum(1).sum(0).cpu()
                    got_ag = True
            del m._probs
        if got_ag:
            rec["attn x grad"] = ag
        else:
            failures["attn x grad"] += 1

        if post:
            rec["rollout (A)"] = rollout(post, ans_rows, img_cols)
            maps, tp, _ = RS.sliced_maps_from_full(
                {l: torch.log(a.clamp_min(1e-12)) for l, a in post.items()}, img_mask, txt_mask)
            isa = torch.tensor([int(p) >= n_prompt for p in tp.tolist()])
            tt  = tokenizer.convert_ids_to_tokens(ids[tp].tolist())
            am  = RS.content_text_mask(tt, tokenizer) & isa
            if int(am.sum()) == 0:
                am = isa
            rec["attention (imp_a)"], *_ = VS.image_importance(maps, am, cand_mask=cand)

        if "_emb" in rec:
            rec["diversity (FPS)"] = farthest_point(rec.pop("_emb"))

        results.append(rec)
        del out, logits, lp, logp, post, store
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"  [skip {i}] {type(e).__name__}: {e}")
        results.append({})
        gc.collect(); torch.cuda.empty_cache()
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{N}")

arms = sorted({k for r in results for k in r})
print(f"\narms computed: {arms}")
if failures:
    print("failures:", dict(failures))

## 4. Score every arm against the LOO ground truth

In [ ]:
def topk_cand(v, k):
    return set(cand_idx[torch.topk(v[cand_idx], k).indices].tolist())

g = torch.Generator().manual_seed(0)
BASE = {"CTRL random": lambda d: torch.rand(L_v, generator=g),
        "gaze proximity": lambda d: (lambda gp: torch.tensor(
            [-math.hypot(i // G - gp // G, i % G - gp % G) for i in range(L_v)]))(
            min(G - 1, int(d["gaze"]["y_norm"] * G)) * G + min(G - 1, int(d["gaze"]["x_norm"] * G)))}

from scipy.stats import norm
for k in KS:
    chance, rows, used = k / n_cand, defaultdict(list), 0
    for d, r in zip(data, results):
        if float(torch.topk(d["drops"][cand_idx], k).values[-1]) <= 1e-6 or not r:
            continue
        used += 1
        gt = topk_cand(d["drops"], k)
        for name, v in list(r.items()) + [(n, f(d)) for n, f in BASE.items()]:
            rows[name].append(len(topk_cand(v, k) & gt) / k)

    print(f"\n=== precision@{k}   chance {chance:.1%}   usable {used}/{N}")
    print(f"{'teacher':<22}{'prec':>8}{'x chance':>10}{'p':>8}")
    print("-" * 48)
    for name in sorted(rows, key=lambda n: -np.mean(rows[n])):
        a = np.array(rows[name]); se = a.std(ddof=1) / max(np.sqrt(len(a)), 1e-9)
        p = 2 * (1 - norm.cdf(abs((a.mean() - chance) / se))) if se > 0 else 1.0
        print(f"{name:<22}{a.mean():>8.1%}{a.mean()/chance:>9.2f}x{p:>8.3f}"
              f"{'  *' if p < 0.05 else ''}")

In [ ]:
# concentration: a label at ~95% of maximum entropy cannot be distilled, whatever its precision
print(f"{'teacher':<22}{'entropy %':>11}{'peak/unif':>11}")
print("-" * 44)
for name in arms:
    ents, peaks = [], []
    for r in results:
        if name not in r:
            continue
        v = r[name][cand_idx].float().clamp_min(0)
        if v.sum() <= 0:
            continue
        p = v / v.sum()
        nz = p[p > 0]
        ents.append(float(-(nz * nz.log()).sum()) / math.log(n_cand))
        peaks.append(float(p.max()) * n_cand)
    if ents:
        print(f"{name:<22}{np.mean(ents):>10.1%}{np.mean(peaks):>11.1f}x")
print("\n(diversity is a rank score, so its entropy is not meaningful)")

## 5. Verdict

* **Any arm at precision@10 >= 35% AND entropy < 85%** -> that is the teacher. Use it at scale and
  spot-check against LOO on a subset, exactly as the FRM spec prescribes (with C substituted for A).
* **Gradient arms clearly beat attention** -> gradients are the right family; the cheaper one
  (~3 forwards/example vs 25-81 for LOO) becomes the scaling path.
* **`grad x input` vs `attn x grad`** -> whichever wins also says which perturbation the label
  should be defined against (embedding-zeroing vs attention-masking). Change the LOO ablation to match.
* **Diversity wins** -> the importance framing is wrong for this model, and the FRM label becomes
  "what spans the scene" rather than "what the answer attended to". This would be the biggest
  surprise and the most publishable.
* **Everything lands ~33% at ~95% entropy** -> no proxy works here. Stop optimising proxies and go
  straight to LOO-at-scale (~35 min for 2500 examples at GROUP=2).